In [6]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import socket, json, time, math, uuid, os
import numpy as np

# Network
LAPTOP_IP         = "192.168.2.1"   # laptop ethernet IP (update if different)
LAPTOP_SHAPE_PORT = 5006            # laptop listens here for recognised shapes
PYNQ_LISTEN_PORT  = 5005            # this PYNQ listens here for incoming strokes

# Shape recognition tuning
RESAMPLE_STEP  = 8.0    # pixels between resampled points
RDP_EPSILON    = 12.0   # RDP simplification tolerance (pixels)
CLOSE_THRESH   = 60.0   # first-to-last distance to count as a closed stroke
CIRCLE_REL_TOL = 0.10   # max rmse/r for circle classification
RECT_ANGLE_TOL = 50.0   # degrees tolerance from 90° at each corner

# JSON output
OUTPUT_DIR = "/home/xilinx/jupyter_notebooks/FPGA/shapes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration OK")
print(f"  PYNQ listens on port {PYNQ_LISTEN_PORT}")
print(f"  Shapes sent to {LAPTOP_IP}:{LAPTOP_SHAPE_PORT}")
print(f"  JSON saved to {OUTPUT_DIR}")

Configuration OK
  PYNQ listens on port 5005
  Shapes sent to 192.168.2.1:5006
  JSON saved to /home/xilinx/jupyter_notebooks/FPGA/shapes


In [7]:
# ── Cell 2: Load FPGA Bitstream ───────────────────────────────────────────────
# Addresses from Vivado Address Editor (confirmed from block design):
#   kasa_0      → 0x43C00000
#   rdp_0       → 0x43C10000
#   rect_detect → 0x43C20000
#   resample_0  → 0x43C30000

import sys
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/FPGA')

from hw_pipeline import (
    load_overlay,
    resample_polyline,
    rdp_simplify,
    kasa_circle_fit,
    try_rectangle
)

load_overlay('/home/xilinx/jupyter_notebooks/FPGA/design_1.bit')
print("FPGA overlay loaded — hardware pipeline ready")

[hw_pipeline] overlay loaded, AXI registers verified, libraries open
FPGA overlay loaded — hardware pipeline ready


In [8]:
# ── Cell 3: Shape Recognition Pipeline ────────────────────────────────────────
# Each step calls the FPGA IP via hw_pipeline.py / libXXX.so

def recognise_shape(stroke_points):
    """
    Full hardware-accelerated shape recognition pipeline.

    Input:  list of [x, y] or (x, y) stroke points in pixel coordinates
    Output: shape dict:
        {'id': str, 'timestamp': float, 'type': 'circle',    'params': {cx, cy, r}}
        {'id': str, 'timestamp': float, 'type': 'rectangle', 'params': {corners}}
        {'id': str, 'timestamp': float, 'type': 'polyline',  'params': {points}}
    """
    pts = [(float(p[0]), float(p[1])) for p in stroke_points]

    if len(pts) < 4:
        return _make_shape('polyline', {'points': pts})

    # Step 1 — Resample to uniform arc-length spacing  [FPGA: resample_0]
    resampled = resample_polyline(pts, RESAMPLE_STEP)

    # Is the stroke closed?
    closed = (len(resampled) >= 10 and
              math.hypot(resampled[0][0] - resampled[-1][0],
                         resampled[0][1] - resampled[-1][1]) < CLOSE_THRESH)
    if closed and resampled[0] != resampled[-1]:
        resampled.append(resampled[0])

    xs    = [p[0] for p in resampled]
    ys    = [p[1] for p in resampled]
    scale = max(20.0, max(max(xs) - min(xs), max(ys) - min(ys)))

    # Step 2 — Kasa circle fit  [FPGA: kasa_0]
    if closed and len(resampled) >= 25:
        fit = kasa_circle_fit(resampled[:-1])   # returns (cx, cy, r) or None
        if fit is not None:
            cx, cy, r = fit
            # rmse computed in software (cheap, just one pass)
            arr  = np.array(resampled[:-1], dtype=float)
            d    = np.sqrt((arr[:,0]-cx)**2 + (arr[:,1]-cy)**2)
            rmse = float(np.sqrt(np.mean((d - r)**2)))
            if 15 <= r <= 0.9*scale and rmse/max(1.0,r) <= CIRCLE_REL_TOL:
                return _make_shape('circle', {
                    'cx': round(float(cx), 2),
                    'cy': round(float(cy), 2),
                    'r':  round(float(r),  2),
                })

    # Step 3 — Rectangle detection  [FPGA: rdp_0 + rect_detect]
    # rdp_simplify first to reduce to corners, then try_rectangle checks angles
    if closed:
        rdp_pts = rdp_simplify(resampled, 0.06 * scale)  # [FPGA: rdp_0]
        corners = try_rectangle(rdp_pts, RECT_ANGLE_TOL)  # [FPGA: rect_detect]
        if corners is not None:
            return _make_shape('rectangle', {
                'corners': [[round(float(p[0]),2), round(float(p[1]),2)] for p in corners]
            })

    # Step 4 — Fallback: RDP-simplified polyline  [FPGA: rdp_0]
    simplified = rdp_simplify(pts, RDP_EPSILON)
    return _make_shape('polyline', {
        'points': [[round(float(p[0]),2), round(float(p[1]),2)] for p in simplified]
    })


def _make_shape(stype, params):
    return {
        'id':        str(uuid.uuid4())[:8],
        'timestamp': time.time(),
        'type':      stype,
        'params':    params,
    }


def save_shape_json(shape):
    path = os.path.join(OUTPUT_DIR, f"{shape['id']}.json")
    with open(path, 'w') as f:
        json.dump(shape, f, indent=2)
    return path


print("Shape pipeline defined")

Shape pipeline defined


In [9]:
# ── Cell 4: Quick Self-Test ───────────────────────────────────────────────────
# Sends known shapes through the pipeline to check the hardware works.

def _circle_pts(cx, cy, r, n=60):
    return [[cx + r*math.cos(2*math.pi*i/n),
             cy + r*math.sin(2*math.pi*i/n)] for i in range(n+1)]

def _rect_pts(x0, y0, w, h, n_per_side=15):
    pts = []
    for i in range(n_per_side): pts.append([x0 + i*w/n_per_side, y0])
    for i in range(n_per_side): pts.append([x0+w, y0 + i*h/n_per_side])
    for i in range(n_per_side): pts.append([x0+w - i*w/n_per_side, y0+h])
    for i in range(n_per_side): pts.append([x0, y0+h - i*h/n_per_side])
    pts.append(pts[0])
    return pts

tests = [
    ('circle',    _circle_pts(320, 240, 80)),
    ('rectangle', _rect_pts(100, 100, 200, 150)),
    ('polyline',  [[100+i*5, 200+i*2] for i in range(30)]),
]

print("Self-test results:")
for expected, pts in tests:
    shape = recognise_shape(pts)
    ok = "✓" if shape['type'] == expected else "✗"
    print(f"  {ok} expected={expected:10s}  got={shape['type']:10s}  params={list(shape['params'].keys())}")

print("\nSelf-test complete")

Self-test results:
  ✓ expected=circle      got=circle      params=['cx', 'cy', 'r']
  ✓ expected=rectangle   got=rectangle   params=['corners']
  ✓ expected=polyline    got=polyline    params=['points']

Self-test complete


In [10]:
import math
import numpy as np

def _rect_pts(x0, y0, w, h, n_per_side=15):
    pts = []
    for i in range(n_per_side): pts.append([x0 + i*w/n_per_side, y0])
    for i in range(n_per_side): pts.append([x0+w, y0 + i*h/n_per_side])
    for i in range(n_per_side): pts.append([x0+w - i*w/n_per_side, y0+h])
    for i in range(n_per_side): pts.append([x0, y0+h - i*h/n_per_side])
    pts.append(pts[0])
    return pts

rect = _rect_pts(100, 100, 200, 150)
resampled = resample_polyline([(float(p[0]),float(p[1])) for p in rect], RESAMPLE_STEP)
fit = kasa_circle_fit(resampled[:-1])
if fit:
    cx, cy, r = fit
    arr = np.array(resampled[:-1], dtype=float)
    d = np.sqrt((arr[:,0]-cx)**2 + (arr[:,1]-cy)**2)
    rmse = float(np.sqrt(np.mean((d-r)**2)))
    print(f"r={r:.1f}  rmse={rmse:.1f}  rmse/r={rmse/r:.3f}  threshold={CIRCLE_REL_TOL}")

r=101.2  rmse=14.8  rmse/r=0.147  threshold=0.1


In [11]:
# ── Cell 5: Benchmark (HW vs SW baseline) ────────────────────────────────────

SW_BASELINE = {
    'resample': 13.805,
    'rdp':      14.009,
    'kasa':      1.522,
    'rect':     13.886,
    'total':    43.221,
}

def benchmark(stroke, runs=100):
    pts = [(float(p[0]), float(p[1])) for p in stroke]

    def time_fn(fn, *args):
        t0 = time.perf_counter()
        for _ in range(runs): fn(*args)
        return (time.perf_counter() - t0) / runs * 1000

    r = resample_polyline(pts, RESAMPLE_STEP)
    hw = {
        'resample': time_fn(resample_polyline, pts, RESAMPLE_STEP),
        'rdp':      time_fn(rdp_simplify,      r,   RDP_EPSILON),
        'kasa':     time_fn(kasa_circle_fit,   r),
        'rect':     time_fn(try_rectangle,     r,   RECT_ANGLE_TOL),
    }
    hw['total'] = sum(hw.values())

    print(f"{'Stage':<22} {'HW (ms)':>10}  {'SW (ms)':>10}  {'Speedup':>8}")
    print("-" * 56)
    for k in ['resample', 'rdp', 'kasa', 'rect', 'total']:
        label = 'Resample' if k=='resample' else k.upper() if k!='total' else 'TOTAL'
        sp = SW_BASELINE[k] / hw[k]
        print(f"  {label:<20} {hw[k]:>10.4f}  {SW_BASELINE[k]:>10.4f}  {sp:>7.2f}x")


test_stroke = [[100 + i*3, 200 + i*2] for i in range(50)]
benchmark(test_stroke)


Stage                     HW (ms)     SW (ms)   Speedup
--------------------------------------------------------
  Resample                11.0761     13.8050     1.25x
  RDP                     13.6767     14.0090     1.02x
  KASA                     1.2279      1.5220     1.24x
  RECT                    45.5303     13.8860     0.30x
  TOTAL                   71.5110     43.2210     0.60x


In [12]:
# ── Cell 6: Live Server Loop ──────────────────────────────────────────────────
# Receives strokes from laptop via UDP, processes them, sends shapes back.
# Stop with the Jupyter kernel stop button (■).

recv_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
recv_sock.bind(('0.0.0.0', PYNQ_LISTEN_PORT))
recv_sock.settimeout(1.0)

send_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
laptop_addr = (LAPTOP_IP, LAPTOP_SHAPE_PORT)

print(f"[pynq_a] listening on port {PYNQ_LISTEN_PORT}")
print(f"[pynq_a] sending shapes to {LAPTOP_IP}:{LAPTOP_SHAPE_PORT}")
print("[pynq_a] running — stop with the Jupyter stop button")

stroke_count = 0

try:
    while True:
        try:
            data, addr = recv_sock.recvfrom(65535)
        except socket.timeout:
            continue

        try:
            packet = json.loads(data.decode('utf-8'))
        except json.JSONDecodeError as e:
            print(f"[pynq_a] bad JSON: {e}")
            continue

        stroke = packet.get('stroke', [])
        if len(stroke) < 2:
            continue

        t0    = time.perf_counter()
        shape = recognise_shape(stroke)
        t_ms  = (time.perf_counter() - t0) * 1000

        path = save_shape_json(shape)
        send_sock.sendto(json.dumps(shape).encode('utf-8'), laptop_addr)

        stroke_count += 1
        print(f"[pynq_a] #{stroke_count:4d}  {shape['type']:12s}  {t_ms:6.2f} ms  → {path}")

except KeyboardInterrupt:
    print("[pynq_a] stopped by user")
finally:
    recv_sock.close()
    send_sock.close()
    print("[pynq_a] sockets closed")

[pynq_a] listening on port 5005
[pynq_a] sending shapes to 192.168.2.1:5006
[pynq_a] running — stop with the Jupyter stop button
[pynq_a] #   1  circle         22.80 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/17b911bc.json
[pynq_a] #   2  circle         23.74 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/95ec11b6.json
[pynq_a] #   3  polyline      199.22 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/0431827a.json
[pynq_a] #   4  rectangle     187.61 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/49e6f0ef.json
[pynq_a] #   5  rectangle     195.36 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/52b5a6c2.json
[pynq_a] #   6  rectangle     142.42 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/a293ae8c.json
[pynq_a] #   7  rectangle     168.45 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/b56b162d.json
[pynq_a] #   8  rectangle     154.98 ms  → /home/xilinx/jupyter_notebooks/FPGA/shapes/27f808f1.json
[pynq_a] #   9  polyline        0.21 ms  → /home/xilinx/jupyter_noteboo

In [ ]:
import subprocess
result = subprocess.run(['ip', 'addr'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import traceback
try:
    import socket, json, time, math, uuid, os
    recv_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    recv_sock.bind(('0.0.0.0', PYNQ_LISTEN_PORT))
    print("Socket bound OK")
    recv_sock.settimeout(1.0)
    data, addr = recv_sock.recvfrom(65535)
    print(f"Got data from {addr}: {data[:50]}")
except Exception as e:
    traceback.print_exc()

In [ ]:
import traceback
try:
    from hw_pipeline import load_overlay, resample_polyline, rdp_simplify, kasa_circle_fit, try_rectangle
    print("imports OK")
    load_overlay('/home/xilinx/jupyter_notebooks/FPGA/design_1.bit')
    print("overlay OK")
except Exception as e:
    traceback.print_exc()

In [ ]:
import socket
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.bind(('0.0.0.0', 5005))
s.settimeout(10.0)
try:
    data, addr = s.recvfrom(65535)
    print(f"✓ Got {len(data)} bytes from {addr}")
except socket.timeout:
    print("✗ Nothing received")
s.close()

In [ ]:
import math
circle_pts = [(int(320 + 80*math.cos(2*math.pi*i/60)),
               int(240 + 80*math.sin(2*math.pi*i/60))) for i in range(60)]

print("Testing resample...")
r = resample_polyline(circle_pts, 8.0)
print(f"  in={len(circle_pts)} out={len(r)}")

print("Testing kasa...")
fit = kasa_circle_fit(r)
print(f"  fit={fit}")

print("Testing rdp...")
s = rdp_simplify(circle_pts, 12.0)
print(f"  in={len(circle_pts)} out={len(s)}")

In [ ]:
# Test resample with just 2 points - should return ~12 points for a 100px line
simple = [(100, 100), (200, 100)]  # horizontal line, 100px long
r = resample_polyline(simple, 8.0)
print(f"2-point line resample: {len(r)} points")
print(f"first few: {r[:5]}")
print(f"last few: {r[-5:]}")

In [ ]:
import os
bit = '/home/xilinx/jupyter_notebooks/FPGA/design_1.bit'
stat = os.stat(bit)
import time
print(f"Bitstream: {stat.st_size} bytes")
print(f"Modified:  {time.ctime(stat.st_mtime)}")

In [ ]:
# Quick logic test - does recognise_shape call try_rectangle correctly?
import inspect
print(inspect.getsource(recognise_shape))

In [13]:
# Paste the points from a circle JSON file here to check its rmse/r
import numpy as np

# Copy the points array from one of your circle JSON files
pts = [
    [991.0, 106.0],  # replace with your actual circle points
    # ...
]

arr = np.array(pts, dtype=float)
x, y = arr[:,0], arr[:,1]
A = np.column_stack([x, y, np.ones_like(x)])
b = x*x + y*y
c, *_ = np.linalg.lstsq(A, b, rcond=None)
cx, cy = 0.5*c[0], 0.5*c[1]
r = np.sqrt(max(1e-9, cx*cx + cy*cy + c[2]))
d = np.sqrt((x-cx)**2 + (y-cy)**2)
rmse = float(np.sqrt(np.mean((d-r)**2)))
print(f"r={r:.1f}  rmse={rmse:.1f}  rmse/r={rmse/r:.3f}")

r=498.3  rmse=0.0  rmse/r=0.000
